# Parte 2 — Clasificación de sentimiento multilingüe

El notebook se guarda en VS Code; el kernel de Colab ejecuta las celdas en un servidor remoto.

1. En **Select Kernel → Colab → New Colab Server**, elige una GPU (por ejemplo T4).
2. En el Explorador, haz clic derecho en la carpeta **data → Upload to Colab**. El notebook busca `/content/data/train.csv` en el servidor y `data/train.csv` cuando se ejecuta desde la raíz local.
3. Ejecuta las celdas en orden. El flujo incluye una prueba inicial, una época de entrenamiento y la generación del primer CSV. Reservamos `eval.csv` para generar las predicciones de Kaggle.

### Carga e inspección de los datos

Leemos `train.csv`, que contiene el identificador, el texto y el sentimiento esperado de cada reseña. Buscamos la carpeta `data` local o su copia en `/content/data` cuando ejecutamos en Colab.

La inspección encontró **20.466 reseñas**, con IDs únicos y sin textos faltantes. Hay 7.486 positivas, 7.381 negativas y 5.599 neutrales. Las neutrales son menos frecuentes, pero para esta referencia no cambiamos la distribución de clases.

In [3]:
from pathlib import Path

import pandas as pd

# Cambia esta ruta si subiste data a otro lugar del servidor.
rutas_data = [Path("data"), Path("/content/data")]
DATA_DIR = next(
    (ruta.resolve() for ruta in rutas_data if (ruta / "train.csv").is_file()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "No encontré train.csv. En el Explorador de VS Code, "
        "haz clic derecho en data → Upload to Colab. "
        "Comprueba que quedó en /content/data."
    )

print("Carpeta de datos:", DATA_DIR)
train = pd.read_csv(DATA_DIR / "train.csv")

print(f"Reseñas: {len(train):,}")
print("Columnas:", train.columns.tolist())
print("IDs duplicados:", train["id"].duplicated().sum())
print("Textos faltantes:", train["text"].isna().sum())

print("\nReseñas por clase:")
print(train["label"].value_counts(dropna=False))

print("\nTres ejemplos:")
for fila in train.sample(3, random_state=42).itertuples(index=False):
    print(f"\nEtiqueta: {fila.label}")
    print(fila.text)

Carpeta de datos: /content/data
Reseñas: 20,466
Columnas: ['id', 'text', 'label']
IDs duplicados: 0
Textos faltantes: 0

Reseñas por clase:
label
positivo    7486
negativo    7381
neutral     5599
Name: count, dtype: int64

Tres ejemplos:

Etiqueta: neutral
Lo pedí a mediados de año y llegó en tres días a mi casa. Lo uso casi a diario en la oficina para juntitas y llamadas. Ya han pasado un par de meses y lo tengo funcionando sin parar. El paquete trae el cable de carga incluido, aparte del manual y la caja normal. Pesa poco y lo guardo en el cajón del escritorio cuando no lo uso.

Etiqueta: positivo
Compré esto a inicios de mes y llegó en unos cuatro días a mi casa. La batería me resultó ser mal construida desde que la estrené. La uso en la oficina casi a diario, más que nada para trámites del trabajo. El manual de uso me pareció estupendo hasta el momento.

Etiqueta: negativo
Lo compre hace un par de meses en una tienda en linea y me llego en tres dias. La duracion de la carga es esp

### Partición de datos

Las 20.466 reseñas tienen IDs únicos y no presentan textos faltantes. Separamos el 80 % para entrenar y el 20 % para validar, con estratificación por sentimiento y semilla 42. Usaremos la misma validación para comparar con y sin aumentación.

La validación queda fuera de las actualizaciones del modelo. La semilla fija el reparto y la estratificación mantiene proporciones parecidas de sentimientos; no garantiza que ambos grupos tengan la misma mezcla de idiomas. El conjunto de evaluación de Kaggle se reserva para generar el envío.

In [4]:
from sklearn.model_selection import train_test_split

SEED = 42

train_fit, val = train_test_split(
    train,
    test_size=0.20,
    random_state=SEED,
    stratify=train["label"],
)

train_fit = train_fit.reset_index(drop=True)
val = val.reset_index(drop=True)

print(f"Entrenamiento: {len(train_fit):,} reseñas")
print(f"Validación: {len(val):,} reseñas")

proporciones = pd.DataFrame({
    "entrenamiento": train_fit["label"].value_counts(normalize=True),
    "validación": val["label"].value_counts(normalize=True),
})

display((proporciones * 100).round(2))

Entrenamiento: 16,372 reseñas
Validación: 4,094 reseñas


,entrenamiento,validación
label,,
positivo,36.57,36.59
negativo,36.07,36.05
neutral,27.36,27.36


### Verificar el entorno

La salida debe mostrar `cuda` y el nombre de la GPU para el primer ajuste. Cambiar de servidor puede requerir volver a subir los datos.

En esta ejecución se detectó una **Tesla T4** y PyTorch seleccionó `cuda`. PyTorch realiza los cálculos de la red; Transformers proporciona el modelo preentrenado y el tokenizador. El notebook está en el Mac, pero el código y los archivos bajo `/content` están en el servidor remoto.

In [5]:
import sys
from importlib.util import find_spec

print("Python del kernel:", sys.executable)
print("Transformers instalado:", find_spec("transformers") is not None)

if find_spec("torch") is None:
    raise RuntimeError("Selecciona el kernel remoto de Colab con GPU.")

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Dispositivo:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Para entrenar, cambia a un servidor de Colab con GPU.")

Python del kernel: /usr/bin/python3
Transformers instalado: True
PyTorch: 2.11.0+cu130
Dispositivo: cuda
GPU: Tesla T4


### Tokenización

Usamos [DistilBERT multilingüe](https://huggingface.co/distilbert/distilbert-base-multilingual-cased) (Apache 2.0). Su tokenizador convierte las reseñas en IDs de palabras y fragmentos de palabras. Conservamos mayúsculas, tildes y puntuación.

La reseña de ejemplo produjo **88 tokens**, incluidos los especiales. Por ejemplo, `pe` y `##dí` representan fragmentos de «pedí». Un `input_id` identifica un token en el vocabulario: no es un peso como en TF-IDF. La secuencia de IDs conserva el orden del texto.

In [6]:
from transformers import AutoTokenizer

MODELO_BASE = "distilbert/distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)

texto = train_fit.iloc[0]["text"]
entrada = tokenizer(texto, return_token_type_ids=False)
ids = entrada["input_ids"]

print("Reseña:", texto)
print("Total de tokens:", len(ids))
display(pd.DataFrame({
    "token": tokenizer.convert_ids_to_tokens(ids[:25]),
    "input_id": ids[:25],
    "attention_mask": entrada["attention_mask"][:25],
}))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Reseña: Lo pedí en oferta después de estar un buen rato comparando precios, llegó en un par de meses porque venía de lejos. Trae carga por un puerto USB tipo C, lo cual es cómodo porque ya tengo cables de ese en casa. Lo uso más que nada en las noches antes de dormir. Y bueno, el botón de encendido superó todas mis expectativas.
Total de tokens: 88


,token,input_id,attention_mask
0,[CLS],101,1
1,Lo,13069,1
2,pe,11161,1
3,##dí,26530,1
4,en,10110,1
5,oferta,56721,1
6,después,12884,1
7,de,10104,1
8,estar,16627,1
9,un,10119,1


### Preparar lotes

El límite de 160 tokens recortó solo 13 de 16.372 reseñas de entrenamiento (0,08 %) en el diagnóstico previo. Añadimos relleno solo hasta el texto más largo de cada lote. DistilBERT no necesita `token_type_ids`.

Las etiquetas se convierten en `negativo = 0`, `neutral = 1` y `positivo = 2`. Son identificadores, no una escala numérica de sentimiento. En el lote de prueba, `input_ids` y `attention_mask` tienen forma `(4, 115)`: cuatro reseñas y 115 posiciones por fila. La máscara marca con 0 el relleno. `labels` tiene forma `(4,)`: una etiqueta por reseña.

In [7]:
from torch.utils.data import Dataset
from transformers import DataCollatorWithPadding

MAX_LENGTH = 160
label2id = {"negativo": 0, "neutral": 1, "positivo": 2}
id2label = {i: etiqueta for etiqueta, i in label2id.items()}


class ResenasDataset(Dataset):
    def __init__(self, tabla):
        self.entradas = tokenizer(
            tabla["text"].tolist(),
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
            return_token_type_ids=False,
        )
        self.etiquetas = tabla["label"].map(label2id).astype(int).tolist()

    def __len__(self):
        return len(self.etiquetas)

    def __getitem__(self, indice):
        ejemplo = {
            nombre: valores[indice]
            for nombre, valores in self.entradas.items()
        }
        ejemplo["labels"] = self.etiquetas[indice]
        return ejemplo


dataset_train = ResenasDataset(train_fit)
dataset_val = ResenasDataset(val)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
lote_prueba = collator([dataset_train[i] for i in range(4)])

for nombre, tensor in lote_prueba.items():
    print(nombre, tuple(tensor.shape))

input_ids (4, 115)
attention_mask (4, 115)
labels (4,)


### Probar el modelo antes de entrenar

Cargamos los pesos preentrenados y una salida nueva para nuestras tres clases. Las puntuaciones iniciales sirven para comprobar el formato, no para evaluar la calidad del clasificador.

La salida `(4, 3)` confirma que hay una puntuación por clase para cada reseña. Estas puntuaciones son *logits*; `softmax` las convierte en probabilidades. El aviso de pesos nuevos en el clasificador es esperado al adaptar un modelo de lenguaje a tres clases. La pérdida de **1,1002** corresponde únicamente al lote de prueba antes del ajuste.

In [8]:
from transformers import AutoModelForSequenceClassification, set_seed

if device.type != "cuda":
    raise RuntimeError("Conecta un kernel Colab con GPU y vuelve a ejecutar el diagnóstico.")

set_seed(SEED)
modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=3,
    label2id=label2id,
    id2label=id2label,
).to(device)

lote_gpu = {nombre: tensor.to(device) for nombre, tensor in lote_prueba.items()}
modelo.eval()
with torch.no_grad():
    salida = modelo(**lote_gpu)

print("Forma de la salida:", tuple(salida.logits.shape))
print(f"Pérdida inicial: {salida.loss.item():.4f}")
display(pd.DataFrame(
    salida.logits.softmax(dim=1).cpu().numpy(),
    columns=[id2label[i] for i in range(3)],
).round(3))

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Forma de la salida: (4, 3)
Pérdida inicial: 1.1002


,negativo,neutral,positivo
0,0.343,0.322,0.335
1,0.348,0.319,0.332
2,0.340,0.329,0.330
3,0.339,0.327,0.334


### Primer ajuste: una época sin aumentación

Entrenamos DistilBERT con las **16.372 reseñas de entrenamiento**. Una época significa recorrerlas una vez: en este caso, 1.024 lotes de hasta 16 reseñas, en orden mezclado.

En cada lote, el modelo produce una puntuación por clase y calcula la **pérdida de entropía cruzada** usando las etiquetas correctas. La retropropagación calcula cómo influyen los parámetros en esa pérdida. AdamW usa esos gradientes para actualizar los pesos con una tasa de aprendizaje de `2e-5`. Ajustamos tanto el clasificador nuevo como el encoder preentrenado.

Usamos precisión mixta para reducir el uso de memoria en la GPU. La validación se ejecuta después del ajuste, sin gradientes ni actualizaciones: contamos los aciertos al elegir la clase con mayor puntuación.

In [9]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

BATCH_SIZE = 16

loader_train = DataLoader(
    dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    generator=torch.Generator().manual_seed(SEED),
)
loader_val = DataLoader(
    dataset_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
)

optimizador = AdamW(modelo.parameters(), lr=2e-5)
scaler = torch.amp.GradScaler("cuda")

modelo.train()
suma_perdida = 0.0
vistos = 0

for paso, lote in enumerate(loader_train, start=1):
    lote = {nombre: tensor.to(device) for nombre, tensor in lote.items()}
    optimizador.zero_grad(set_to_none=True)

    # Usamos precisión mixta para ahorrar memoria en la GPU.
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        perdida = modelo(**lote).loss

    scaler.scale(perdida).backward()
    scaler.step(optimizador)
    scaler.update()

    n = lote["labels"].size(0)
    suma_perdida += perdida.item() * n
    vistos += n

    if paso % 100 == 0 or paso == len(loader_train):
        print(
            f"Lote {paso}/{len(loader_train)} | "
            f"Pérdida media: {suma_perdida / vistos:.4f}",
            flush=True,
        )

print("\nEvaluando en validación...", flush=True)
modelo.eval()
aciertos = 0
vistos_val = 0

with torch.inference_mode():
    for lote in loader_val:
        lote = {nombre: tensor.to(device) for nombre, tensor in lote.items()}
        etiquetas = lote.pop("labels")
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = modelo(**lote).logits

        aciertos += (logits.argmax(dim=1) == etiquetas).sum().item()
        vistos_val += etiquetas.size(0)

accuracy_val = aciertos / vistos_val
print(f"Accuracy de validación: {accuracy_val:.4f} ({aciertos}/{vistos_val})")

Lote 100/1024 | Pérdida media: 0.8407
Lote 200/1024 | Pérdida media: 0.7288
Lote 300/1024 | Pérdida media: 0.6778
Lote 400/1024 | Pérdida media: 0.6296
Lote 500/1024 | Pérdida media: 0.5892
Lote 600/1024 | Pérdida media: 0.5562
Lote 700/1024 | Pérdida media: 0.5293
Lote 800/1024 | Pérdida media: 0.5052
Lote 900/1024 | Pérdida media: 0.4848
Lote 1000/1024 | Pérdida media: 0.4693
Lote 1024/1024 | Pérdida media: 0.4672

Evaluando en validación...
Accuracy de validación: 0.8385 (3433/4094)


### Resultado de validación

La pérdida media acumulada de entrenamiento al final de la época fue **0,4672**. En validación, el modelo acertó **3.433 de 4.094 reseñas**, para un *accuracy* de **0,8385 (83,85 %)**. La pérdida y el *accuracy* miden cosas distintas: la primera penaliza las probabilidades asignadas a las etiquetas correctas; el segundo cuenta etiquetas acertadas.

Esta es la referencia **sin aumentación** en nuestra partición local. No es un resultado de Kaggle ni permite concluir, con una sola época, si entrenar más mejorará.

### Guardar el modelo y generar el envío

Guardamos los pesos, la configuración y el tokenizador para poder recargar el modelo. Luego usamos `eval.csv` solo para inferencia, con el mismo tokenizador y límite de 160 tokens. Convertimos la clase numérica de mayor puntuación en `negativo`, `neutral` o `positivo`.

Construimos el CSV en el orden de `sample_submission.csv`, asociando las predicciones por `id`. Comprobamos las columnas, los identificadores, las etiquetas y las **6.781 filas** requeridas.

In [10]:
from pathlib import Path

# Guardar los pesos, la configuración de clases y el tokenizador.
ruta_modelo = Path("/content/modelos/distilbert_multilingue_epoca1")
modelo.save_pretrained(ruta_modelo)
tokenizer.save_pretrained(ruta_modelo)
print("Modelo guardado en:", ruta_modelo, flush=True)

evaluacion = pd.read_csv(DATA_DIR / "eval.csv")
plantilla = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert list(plantilla.columns) == ["id", "answer"]
assert evaluacion["id"].is_unique and plantilla["id"].is_unique
assert set(evaluacion["id"]) == set(plantilla["id"])
assert evaluacion["text"].notna().all()

modelo.eval()
predicciones = []

with torch.inference_mode():
    for inicio in range(0, len(evaluacion), BATCH_SIZE):
        textos = evaluacion["text"].iloc[inicio:inicio + BATCH_SIZE].tolist()
        lote = tokenizer(
            textos,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_token_type_ids=False,
            return_tensors="pt",
        ).to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = modelo(**lote).logits

        predicciones.extend(logits.argmax(dim=1).cpu().tolist())

# Asociar cada predicción a su ID y respetar el orden de la plantilla.
pred_por_id = pd.Series(
    [id2label[i] for i in predicciones],
    index=evaluacion["id"],
)
submission = plantilla[["id"]].copy()
submission["answer"] = submission["id"].map(pred_por_id)

assert len(submission) == 6781
assert submission["answer"].isin(label2id).all()

ruta_csv = Path("/content/submissions/submission_distilbert_epoca1.csv")
ruta_csv.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(ruta_csv, index=False)

display(submission.head())
print("\nPredicciones por clase:")
print(submission["answer"].value_counts())
print(f"\nCSV listo: {ruta_csv} ({len(submission):,} filas)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo guardado en: /content/modelos/distilbert_multilingue_epoca1


,id,answer
0,0,negativo
1,1,neutral
2,2,negativo
3,3,neutral
4,4,negativo



Predicciones por clase:
answer
negativo    3416
positivo    2068
neutral     1297
Name: count, dtype: int64

CSV listo: /content/submissions/submission_distilbert_epoca1.csv (6,781 filas)


### Archivos generados

El CSV contiene 3.416 predicciones negativas, 2.068 positivas y 1.297 neutrales. Estos conteos describen las predicciones; no indican cuántas son correctas.

El modelo quedó en `/content/modelos/distilbert_multilingue_epoca1` y el envío en `/content/submissions/submission_distilbert_epoca1.csv`. Son rutas del servidor de Colab. Para conservarlos fuera de la sesión, se pueden descargar desde la vista de archivos de Colab en VS Code o copiar a Drive.